# European Streetview CNN Classifier
Trains a CNN to predict the European country from 640x640 streetview images.
- **Train set**: 75% of `trainEurope/` images (stratified per country folder)
- **Validation set**: 25% of `trainEurope/` images
- **Test set**: all of `testEurope/`

In [ ]:
# ── 1. Imports & reproducibility ────────────────────────────────────────────
import os, random, json
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# ── 2. Configuration ─────────────────────────────────────────────────────────
TRAIN_DIR   = 'trainEurope'   # adjust path if needed
TEST_DIR    = 'testEurope'
IMG_SIZE    = 640             # images are already 640×640
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-4
VAL_SPLIT   = 0.25            # 25 % of each country for validation
NUM_WORKERS = 4

# Augmentation / normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
# ── 3. Dataset definition ────────────────────────────────────────────────────
class StreetviewDataset(Dataset):
    """Loads images from a directory tree: root/<country>/<img>.jpg"""

    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []   # list of (path, label_idx)

        countries = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        # Build or reuse class→index mapping
        if class_to_idx is None:
            self.class_to_idx = {c: i for i, c in enumerate(countries)}
        else:
            self.class_to_idx = class_to_idx

        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.classes = countries

        IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
        for country in countries:
            if country not in self.class_to_idx:
                continue
            label = self.class_to_idx[country]
            folder = os.path.join(root_dir, country)
            for fname in os.listdir(folder):
                if os.path.splitext(fname)[1].lower() in IMG_EXTS:
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


def stratified_split(dataset, val_fraction=0.25, seed=42):
    """Returns train / val index lists keeping val_fraction per class."""
    labels = [s[1] for s in dataset.samples]
    indices = list(range(len(labels)))
    train_idx, val_idx = train_test_split(
        indices,
        test_size=val_fraction,
        stratify=labels,
        random_state=seed
    )
    return train_idx, val_idx


# Build full training dataset (no transform yet – transforms are applied via Subset wrappers)
full_train_ds = StreetviewDataset(TRAIN_DIR, transform=None)
NUM_CLASSES   = len(full_train_ds.class_to_idx)
print(f'Countries found: {NUM_CLASSES}')
print(f'Total training images: {len(full_train_ds)}')
print('Classes:', list(full_train_ds.class_to_idx.keys())[:10], '...')

In [ ]:
# ── 4. Build train / val splits ──────────────────────────────────────────────
class TransformSubset(Dataset):
    """Wraps a Subset and applies a transform at access time."""
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


train_idx, val_idx = stratified_split(full_train_ds, val_fraction=VAL_SPLIT, seed=SEED)

train_subset = Subset(full_train_ds, train_idx)
val_subset   = Subset(full_train_ds, val_idx)

train_ds = TransformSubset(train_subset, train_tf)
val_ds   = TransformSubset(val_subset,   val_tf)

print(f'Train samples : {len(train_ds)}')
print(f'Val   samples : {len(val_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# ── 5. Model (EfficientNet-B3 fine-tuned) ───────────────────────────────────
def build_model(num_classes, freeze_backbone=False):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace classifier head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )
    return model


model = build_model(NUM_CLASSES).to(DEVICE)
print(model.classifier)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
# ── 6. Loss, optimiser, scheduler ───────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [ ]:
# ── 7. Training loop ─────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)

    return total_loss / total, correct / total


print(f'{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}')
print('-' * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader,   training=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'best_model.pth')

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>8.2%} | {vl_loss:>8.4f} | {vl_acc:>6.2%}')

print(f'\nBest validation accuracy: {best_val_acc:.2%}')

In [ ]:
# ── 8. Loss & accuracy curves ────────────────────────────────────────────────
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   linewidth=2, linestyle='--')
axes[0].set_title('Loss Curve', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, [a * 100 for a in history['train_acc']], label='Train Acc', linewidth=2)
axes[1].plot(epochs, [a * 100 for a in history['val_acc']],   label='Val Acc',   linewidth=2, linestyle='--')
axes[1].set_title('Accuracy Curve', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 – European Streetview Classification', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Curves saved to training_curves.png')

In [ ]:
# ── 9. Evaluate on testEurope ────────────────────────────────────────────────
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval()

# Build test dataset reusing the same class_to_idx mapping
test_ds = StreetviewDataset(
    TEST_DIR,
    transform=val_tf,
    class_to_idx=full_train_ds.class_to_idx
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f'Test images: {len(test_ds)}')

all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f'\nTest Accuracy: {test_acc:.2%}')

In [ ]:
# ── 10. Per-country classification report ───────────────────────────────────
class_names = [full_train_ds.idx_to_class[i] for i in range(NUM_CLASSES)]

print(classification_report(
    all_labels, all_preds,
    target_names=class_names,
    digits=3
))

In [ ]:
# ── 11. Confusion matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig_h = max(10, NUM_CLASSES // 2)
plt.figure(figsize=(fig_h + 2, fig_h))
sns.heatmap(
    cm, annot=(NUM_CLASSES <= 20),
    fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion Matrix – testEurope', fontsize=14)
plt.ylabel('True Country')
plt.xlabel('Predicted Country')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved to confusion_matrix.png')

In [ ]:
# ── 12. Quick inference helper ───────────────────────────────────────────────
def predict_image(image_path, model, class_to_idx, transform, device, top_k=5):
    """Return top-k predicted countries for a single image."""
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]

    top_probs, top_indices = torch.topk(probs, k=top_k)
    results = [(idx_to_class[i.item()], p.item()) for i, p in zip(top_indices, top_probs)]

    # Display
    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                              gridspec_kw={'width_ratios': [1, 2]})
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Input Image')

    countries = [r[0] for r in results]
    confidences = [r[1] * 100 for r in results]
    bars = axes[1].barh(countries[::-1], confidences[::-1], color='steelblue')
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title(f'Top-{top_k} Predictions')
    axes[1].set_xlim(0, 100)
    for bar, conf in zip(bars, confidences[::-1]):
        axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                     f'{conf:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    return results


# Example usage – replace with any image path:
# predict_image('testEurope/France/img_001.jpg', model, full_train_ds.class_to_idx, val_tf, DEVICE)